# 02 · RSSI representations and feature checks

**Owner: Member 2 (Features and representation lead).** Use training folds only for every fitted transform. `validationData.csv` remains final-evaluation-only.

In [ ]:
from pathlib import Path
import sys, json
ROOT=Path.cwd()
if not (ROOT / "src").exists(): ROOT=Path.cwd().parent
sys.path.insert(0,str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from src.data.load import load_data
from src.features.transform import build_rssi_features, build_engineered_features, FoldFeatureBuilder
train, official_test, quality=load_data()
print(f"clean train={len(train):,}; untouched official test={len(official_test):,}")

## One missing-value convention

The source sentinel `+100` means “not detected”, not a measured strength. All RSSI views first map it to -110 dBm; transformed signal features encode that as zero and the missingness mask separately marks absence.

In [ ]:
sample=train.iloc[:5]
views={name: build_rssi_features(sample,name) for name in ("linear_positive","exponential","powed")}
pd.concat([views[k].iloc[:, :4].add_prefix(k+"_") for k in views],axis=1).head()

## Engineered fingerprint features

In [ ]:
engineered=build_engineered_features(sample,top_k=3)
print(engineered.shape)
engineered.iloc[:,:11]

## Representation definitions

In [ ]:
definitions=pd.DataFrame([
 ["linear_positive", "(RSSI + 110) / 110", "Linear signal scale, range [0, 1]"],
 ["exponential", "exp(RSSI / 24)", "Exponential signal transform"],
 ["powed", "((RSSI + 110) / 110) ** e", "Powed; beta=e"],
],columns=["view","observed signal mapping","interpretation"])
display(definitions)

## PCA retained variance on one training fold

In [ ]:
folds=json.loads((ROOT/"data/splits/folds.json").read_text())["folds"]
fold=folds[0]; fit=train.iloc[fold["train_idx"]]
builder=FoldFeatureBuilder(representation="linear_positive",include_engineered=True,include_missingness=True)
x=builder.fit_transform(fit)
pca=PCA(n_components=200,random_state=42).fit(x)
evr=pca.explained_variance_ratio_
pca_summary=pd.DataFrame({"components":[50,100,200],"cumulative_explained_variance":[evr[:k].sum() for k in (50,100,200)]})
display(pca_summary)
plt.figure(figsize=(7,4)); plt.plot(np.arange(1,201),np.cumsum(evr)); plt.xlabel("PCA components"); plt.ylabel("Cumulative explained variance"); plt.grid(True); plt.tight_layout()

## Variance threshold and RF/XGBoost handoff

The fitted feature builder supports `variance_threshold=0.0`; the selector is fit on each training fold and reused on validation/test. Member 3 owns the RF/XGBoost model comparisons, so tree-score deltas should be attached to the shared tree experiment when that model module is ready. This notebook provides the same fold-safe feature selection point for that handoff.

In [ ]:
selector=VarianceThreshold(threshold=0.0).fit(x)
print(f"before={x.shape[1]:,} features; after={selector.transform(x).shape[1]:,} non-constant features")

## Grouped-CV kNN comparison

Run `python scripts/run_knn_baselines.py` to compare building, floor, and coordinate predictions across all five USERID-held-out folds for the three representations. Each fold fits its scaler only on that fold’s training rows. The official test set is not used.